# 04 · Agent reasoning — known-mechanism recovery, ablation, calibration

Three experiments that validate the **reasoning layer** directly:
- **Recovery** — on 11 characterized variants, does it recover the established TF/gene/trait?
- **Ablation** — single vs multi-agent−redteam vs full: what does the architecture buy?
- **Calibration** — does confidence track evidence strength across strata?

Coordinates are resolved from rsIDs via Ensembl (no hand-typed positions). Needs scorer + key.

In [ ]:
# --- Setup: GPU + install RegLens -------------------------------------------
# Upload reglens_for_colab.zip via the Files panel first (git archive of the repo).
!nvidia-smi -L || echo 'NO GPU — set Runtime > Change runtime type > GPU.'
!pip -q install 'tensorflow>=2.12' pyfaidx typer numpy matplotlib
!unzip -oq /content/reglens_for_colab.zip -d /content/RegLens && pip -q install -e /content/RegLens
import os; os.chdir('/content/RegLens'); print('cwd', os.getcwd())

In [ ]:
# --- Reference genome (hg38) + pretrained K562 ChromBPNet model -------------
import glob, os
os.makedirs('/content/ref', exist_ok=True)
%cd /content/ref
# hg38 via the Broad public bucket: resolves on Colab, chr-named, ships its .fai.
!wget -c -O hg38.fa     https://storage.googleapis.com/gcp-public-data--broad-references/hg38/v0/Homo_sapiens_assembly38.fasta
!wget -c -O hg38.fa.fai https://storage.googleapis.com/gcp-public-data--broad-references/hg38/v0/Homo_sapiens_assembly38.fasta.fai
# ENCODE K562 ATAC ChromBPNet models (5 folds, bias-corrected).
![ -f ENCFF984RAF.tar.gz ] || wget -q -c -O ENCFF984RAF.tar.gz https://www.encodeproject.org/files/ENCFF984RAF/@@download/ENCFF984RAF.tar.gz
!mkdir -p encode_models && tar -xzf ENCFF984RAF.tar.gz -C encode_models
n = len(glob.glob('encode_models/**/*chrombpnet_nobias*.h5', recursive=True))
print(n, 'K562 folds | genome:', os.path.exists('hg38.fa'))
%cd /content/RegLens

In [ ]:
# --- Reasoning layer: Anthropic SDK + API key -------------------------------
!pip -q install anthropic
import os
os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'   # <-- paste your key
assert os.environ['ANTHROPIC_API_KEY'].startswith('sk-'), 'set your key above'

In [ ]:
# Fold + reverse-complement averaged K562 backend (the validated engine).
from reglens.tools.chrombpnet_score import ChromBPNetScorer, load_backend
scorer = ChromBPNetScorer(load_backend('/content/ref/encode_models', average_rc=True),
                          window_length=2114, model_name='K562-5fold+rc')
GENOME = '/content/ref/hg38.fa'

In [ ]:
# --- Recovery: does the agent recover the known TF / gene / trait? -----------
from reglens.agents.multi_agent import MultiAgentInterpreter
from reglens.validation import agent_eval as ae
rec = ae.run_recovery(MultiAgentInterpreter(), scorer=scorer, genome_path=GENOME, progress=True)
print(ae.render_recovery(rec)); print(ae.recovery_rates(rec))
for r in rec:
    print('\n', r.known.rsid, r.known.gene, r.known.tf, '| conf', r.confidence)
    print('  ', r.interpretation.mechanism)

In [ ]:
# --- Ablation: single vs multi−redteam vs full, over the SAME bundle ---------
from reglens.agents.interpreter import ClaudeInterpreter
from reglens.validation.dataset import load_labeled_variants
from reglens.validation.null_control import select_negatives
HEMA = ["BCL11A","HBB","HBG1","PKLR-48h","PKLR-24h","GP1BA"]
strong = [(ae.resolve_variant(km.rsid, prefer_alt=km.alt), km.rsid, km.rsid, 'strong')
          for km in ae.KNOWN_MECHANISMS[:4]]
nulls  = [(v.variant, v.rsid, v.source, 'null')
          for v in select_negatives(load_labeled_variants('/content/RegLens/data/benchmarks/kircher_mpra_grch38.tsv'),
                                     n=4, seed=7, elements=HEMA)]
rows = ae.run_ablation(strong + nulls,
    single=ClaudeInterpreter(), multi_no_redteam=MultiAgentInterpreter(redteam=False),
    multi_full=MultiAgentInterpreter(redteam=True),
    scorer=scorer, genome_path=GENOME, progress=True)
print(ae.render_ablation(rows))

In [ ]:
# --- Calibration: confidence (high/med/low) x stratum -----------------------
from reglens.validation.null_control import run_paired_control
neg, weak = run_paired_control('/content/RegLens/data/benchmarks/kircher_mpra_grch38.tsv', MultiAgentInterpreter(),
    n_neg=4, n_pos=4, seed=7, elements=HEMA, celltype='K562',
    genome_path=GENOME, scorer=scorer, progress=True)
strata = {'strong': [r.interpretation for r in rec],
          'weak':   [o.interpretation for o in weak],
          'null':   [o.interpretation for o in neg]}
print(ae.render_calibration(ae.calibration_table(strata)))

## Honest reading

Recovery: trait 11/11, gene 10/11, TF 8/11 — the TF misses are anti-confabulation (it
refused to name a TF the motif tool didn't surface, even when it knew the answer).
Ablation: the multi-agent + red-team only ever *lower* overconfidence, never raise it.
Calibration: `medium+` confidence appears only in the strong stratum. See `RESULTS.md`.